# Autocorrelación Espacial Global (*Global Spatial Autocorrelation*)

**Adaptado de:** Rey, Arribas-Bel & Wolf – *Geographic Data Science with Python*, Capítulo 6.

# 1. Entendiendo la autocorrelación espacial

La **autocorrelación espacial** se refiere a la existencia de una relación funcional entre lo que ocurre en un punto del espacio y lo que ocurre en otros lugares. El concepto examina cómo la similitud en los **valores** de una variable se correlaciona con la similitud en las **ubicaciones** geográficas.

A diferencia de la autocorrelación temporal, la autocorrelación espacial conecta los valores de una variable en una ubicación dada con los valores de la **misma variable** en otras ubicaciones.

La **aleatoriedad espacial** representa una condición base donde la ubicación no proporciona información sobre el valor de la variable. La autocorrelación espacial se define como la **ausencia de aleatoriedad espacial**.

La autocorrelación espacial se manifiesta en dos dimensiones: **signo** y **escala**.

### Autocorrelación espacial positiva

Ocurre cuando valores similares se **agrupan** geográficamente. Por ejemplo, en la distribución del ingreso, las áreas de altos ingresos tienden a estar cerca de otras áreas de altos ingresos, y la pobreza se concentra en regiones específicas.

### Autocorrelación espacial negativa

Ocurre cuando valores similares se distribuyen **lejos unos de otros**. Aparece en escenarios de competencia espacial, como la localización de instalaciones (supermercados, hospitales).

### Global vs. Local

- **Autocorrelación global:** examina las tendencias espaciales generales. ¿Siguen los valores patrones geográficos particulares? ¿Están los valores similares más cerca de lo que el azar predeciría?
- **Autocorrelación local:** se enfoca en desviaciones de las tendencias globales a escalas más pequeñas.

En este notebook utilizaremos herramientas de **Análisis Exploratorio de Datos Espaciales** (ESDA), que son análogas al análisis exploratorio de datos tradicional, pero con la **geografía en el centro del análisis**.

# 2. Ilustración empírica: el Referéndum de la UE (Brexit)

El **referéndum del Brexit** de 2016 en el Reino Unido es nuestro ejemplo empírico. Combinamos los datos de la **Comisión Electoral** (porcentajes de voto a nivel de autoridad local) con los límites geográficos de los **Distritos de Autoridad Local** (LAD) de la ONS.

La variable principal que analizaremos es `Pct_Leave`, que mide la **proporción de votos a favor de abandonar la UE**.

## 2.1 Configuración e importación de librerías

In [ ]:
# Gráficos
import matplotlib.pyplot as plt
import seaborn
import splot
from splot.esda import plot_moran
import contextily

# Análisis
import geopandas
import pandas
import esda
from libpysal import weights
from numpy.random import seed

## 2.2 Carga y preparación de datos

In [ ]:
# Cargar resultados del referéndum (CSV)
brexit_data_path = "datos/external/brexit/brexit_vote.csv"
ref = pandas.read_csv(brexit_data_path, index_col="Area_Code")
ref.info()

In [ ]:
# Cargar límites geográficos de los distritos de autoridad local (GeoJSON)
lads = geopandas.read_file(
    "datos/external/brexit/local_authority_districts.geojson"
).set_index("lad16cd")
lads.info()

In [ ]:
# Combinar datos electorales con geometrías
db = (
    geopandas.GeoDataFrame(
        lads.join(ref[["Pct_Leave"]]), crs=lads.crs
    )
    .to_crs(epsg=3857)[
        ["objectid", "lad16nm", "Pct_Leave", "geometry"]
    ]
    .dropna()
)
db.info()

## 2.3 Mapa coroplético del % Leave

In [ ]:
f, ax = plt.subplots(1, figsize=(9, 9))
db.plot(
    column="Pct_Leave",
    cmap="viridis",
    scheme="quantiles",
    k=5,
    edgecolor="white",
    linewidth=0.0,
    alpha=0.75,
    legend=True,
    legend_kwds={"loc": 2},
    ax=ax,
)
contextily.add_basemap(
    ax,
    crs=db.crs,
    source=contextily.providers.CartoDB.Positron,
)
ax.set_axis_off()

El mapa sugiere la presencia de **autocorrelación espacial positiva**: los porcentajes altos de Leave se agrupan en las regiones del este de Inglaterra, mientras que los porcentajes bajos se concentran en Escocia y Londres.

Sin embargo, los seres humanos somos muy buenos detectando patrones — incluso en datos aleatorios. Los mapas coropléticos pueden distorsionar la percepción a través de variaciones en forma y tamaño. Por eso necesitamos **estadísticos formales** para evaluar si los patrones observados superan lo que el azar produciría.

## 2.4 Construcción de la matriz de pesos espaciales

In [ ]:
# Generar W de k-vecinos más cercanos (k=8)
w = weights.KNN.from_dataframe(db, k=8)
# Estandarización por filas
w.transform = "R"

# 3. Autocorrelación espacial global

Los indicadores de autocorrelación espacial global resumen la distribución espacial de los valores y cuantifican las desviaciones respecto a la aleatoriedad. Estos estadísticos caracterizan el **grado de agrupamiento** (*clustering*) y proporcionan resúmenes visuales o numéricos.

## 3.1 Rezago espacial (*Spatial Lag*)

El **operador de rezago espacial** es fundamental en el análisis espacial. Aplica una matriz de pesos a una variable para calcular un promedio ponderado de los valores vecinos.

**Notación matricial:**

$$Y_{sl} = W \cdot Y$$

**Notación individual:**

$$y_{sl-i} = \sum_j w_{ij} \cdot y_j$$

Donde $w_{ij}$ representa la celda en $W$ para la fila $i$ y columna $j$, capturando la relación espacial entre las observaciones $i$ y $j$. El valor $y_{sl-i}$ captura los productos de valores y pesos para los vecinos de $i$ (los no-vecinos reciben peso cero).

- Con pesos **binarios**, el rezago espacial suma los valores de los vecinos.
- Con pesos **estandarizados por fila** (acotados entre 0 y 1), representa el **promedio local** — el valor promedio de $Y$ en el vecindario de la observación $i$.

In [ ]:
# Calcular el rezago espacial del % Leave
db["Pct_Leave_lag"] = weights.spatial_lag.lag_spatial(
    w, db["Pct_Leave"]
)

In [ ]:
# Comparar valores originales vs. rezago para Liverpool y Midlothian
db.loc[["E08000012", "S12000019"], ["Pct_Leave", "Pct_Leave_lag"]]

**Liverpool** (E08000012) muestra un patrón notable: como región "Remain" en medio del norte de Inglaterra que votó Leave, se destaca. Con ~42% de voto Leave, su rezago espacial (~57%) refleja los porcentajes más altos de sus vecinos.

**Midlothian** (S12000019) en Escocia tiene ~38% de Leave pero un rezago de ~28%, reflejando la fuerte posición Remain de Escocia.

A pesar de tener porcentajes individuales similares, sus **contextos espaciales diferentes** producen rezagos espaciales distintos. El rezago espacial suaviza las variaciones locales.

In [ ]:
# Visualización lado a lado: variable original vs. rezago espacial
f, axs = plt.subplots(1, 2, figsize=(12, 6))
ax1, ax2 = axs

db.plot(
    column="Pct_Leave",
    cmap="viridis",
    scheme="quantiles",
    k=5,
    edgecolor="white",
    linewidth=0.0,
    alpha=0.75,
    legend=True,
    ax=ax1,
)
ax1.set_axis_off()
ax1.set_title("% Leave")
contextily.add_basemap(
    ax1,
    crs=db.crs,
    source=contextily.providers.CartoDB.Positron,
)

db.plot(
    column="Pct_Leave_lag",
    cmap="viridis",
    scheme="quantiles",
    k=5,
    edgecolor="white",
    linewidth=0.0,
    alpha=0.75,
    legend=True,
    ax=ax2,
)
ax2.set_axis_off()
ax2.set_title("% Leave - Rezago Espacial")
contextily.add_basemap(
    ax2,
    crs=db.crs,
    source=contextily.providers.CartoDB.Positron,
)

plt.show()

El rezago espacial suaviza las diferencias abruptas del mapa original. Las zonas atípicas (como Liverpool, aislada en medio de vecinos con alto % Leave) se atenúan en el mapa de la derecha.

## 3.2 Caso binario: *Join Counts* (conteo de uniones)

Los estadísticos de **Join Counts** evalúan la autocorrelación espacial para **variables binarias**. Una autoridad local que votó Leave se codifica como 1, y como 0 en caso contrario.

El estadístico conceptualiza un tablero con cuadrados de dos colores (verde = 0, amarillo = 1) y cuenta las uniones entre pares vecinos:

| Tipo de unión | Descripción | Autocorrelación |
|---------------|-------------|------------------|
| GG (verde-verde) | Vecinos con el mismo valor (0-0) | Positiva |
| AA (amarillo-amarillo) | Vecinos con el mismo valor (1-1) | Positiva |
| GA (verde-amarillo) | Vecinos con distinto valor (0-1) | Negativa |

Si observamos **más** uniones GG/AA de lo esperado bajo aleatoriedad, hay autocorrelación **positiva**. Si hay más GA, es **negativa**.

In [ ]:
# Crear variable binaria: 1 si el % Leave > 50
db["Leave"] = (db["Pct_Leave"] > 50).astype(int)
db[["Pct_Leave", "Leave"]].tail()

In [ ]:
# Mapa de la variable binaria
f, ax = plt.subplots(1, figsize=(9, 9))
db.plot(
    ax=ax,
    column="Leave",
    categorical=True,
    legend=True,
    edgecolor="0.5",
    linewidth=0.25,
    cmap="Set3",
    figsize=(9, 9),
)
ax.set_axis_off()
ax.set_title("Mayoría Leave")
plt.axis("equal")
plt.show()

In [ ]:
# Join Counts requiere pesos binarios (no estandarizados)
w.transform = "O"

In [ ]:
# Calcular Join Counts
seed(1234)
jc = esda.join_counts.Join_Counts(db["Leave"], w)

In [ ]:
# Resultados: conteo de uniones observadas
print(f"Uniones GG (0-0): {jc.bb}")
print(f"Uniones AA (1-1): {jc.ww}")
print(f"Uniones GA (0-1): {jc.bw}")
print(f"Total de uniones: {jc.J}")
print(f"Verificación: {jc.bb + jc.ww + jc.bw}")

In [ ]:
# Valores esperados bajo aleatoriedad espacial
print(f"Uniones GG esperadas: {jc.mean_bb:.1f}")
print(f"Uniones GA esperadas: {jc.mean_bw:.1f}")

In [ ]:
# Pseudo p-valores (prueba de significancia mediante permutaciones)
print(f"P-valor para uniones misma categoría (GG): {jc.p_sim_bb:.4f}")
print(f"P-valor para uniones distinta categoría (GA): {jc.p_sim_bw:.4f}")

PySAL realiza **999 permutaciones espaciales** bajo la hipótesis nula de aleatoriedad, generando pseudo p-valores. Los resultados demuestran una fuerte **autocorrelación espacial positiva**: ocurren significativamente más uniones de misma categoría de lo esperado (`p_sim_bb`), y significativamente menos uniones de categoría opuesta (`p_sim_bw`).

## 3.3 Caso continuo: Gráfico de Moran e I de Moran

Para variables **continuas**, el estadístico más utilizado es el **I de Moran**:

$$I = \frac{n}{\sum_i \sum_j w_{ij}} \cdot \frac{\sum_i \sum_j w_{ij} \, z_i \, z_j}{\sum_i z_i^2}$$

Donde:
- $n$ = número de observaciones
- $z_i = y_i - \bar{y}$ es el valor estandarizado (centrado en la media)
- $w_{ij}$ es la celda de la matriz de pesos para la fila $i$ y columna $j$

### Gráfico de Moran (*Moran Plot*)

El gráfico de Moran visualiza la autocorrelación espacial mostrando la variable estandarizada en el eje X contra su **rezago espacial estandarizado** en el eje Y. La **pendiente de la línea de regresión** ajustada corresponde al I de Moran.

In [ ]:
# Estandarizar la variable (restar la media)
db["Pct_Leave_std"] = db["Pct_Leave"] - db["Pct_Leave"].mean()

# Calcular el rezago espacial de la variable estandarizada
db["Pct_Leave_lag_std"] = weights.lag_spatial(
    w, db["Pct_Leave_std"]
)

In [ ]:
# Gráfico de Moran
f, ax = plt.subplots(1, figsize=(6, 6))
seaborn.regplot(
    x="Pct_Leave_std",
    y="Pct_Leave_lag_std",
    ci=None,
    data=db,
    line_kws={"color": "r"},
)
ax.axvline(0, c="k", alpha=0.5)
ax.axhline(0, c="k", alpha=0.5)
ax.set_title("Gráfico de Moran - % Leave")
plt.show()

El gráfico muestra una **relación positiva** entre el porcentaje estandarizado de Leave y su rezago espacial (interpretado como el promedio del vecindario gracias a la estandarización por filas). El ajuste lineal incluido representa la mejor línea recta de esta relación.

Una relación positiva indica **autocorrelación espacial positiva**: valores similares se agrupan geográficamente. Los valores altos tienden a estar cerca de otros valores altos; los bajos cerca de otros bajos.

Los cuatro cuadrantes del gráfico tienen interpretaciones específicas:

| Cuadrante | Valor propio | Valor vecinos | Interpretación |
|-----------|-------------|---------------|----------------|
| Superior derecho (I) | Alto | Alto | Cluster de valores altos (HH) |
| Inferior izquierdo (III) | Bajo | Bajo | Cluster de valores bajos (LL) |
| Superior izquierdo (II) | Bajo | Alto | Atípico espacial (LH) |
| Inferior derecho (IV) | Alto | Bajo | Atípico espacial (HL) |

In [ ]:
# Calcular I de Moran formalmente
w.transform = "R"
moran = esda.moran.Moran(db["Pct_Leave"], w)

In [ ]:
# Valor del estadístico
print(f"I de Moran: {moran.I:.4f}")

In [ ]:
# Pseudo p-valor (significancia estadística)
print(f"Pseudo p-valor: {moran.p_sim:.4f}")

El p-valor indica que, bajo aleatoriedad espacial, solo el 0.01% de los mapas generados mostrarían valores absolutos de I de Moran más grandes que el observado. La distribución espacial observada es **significativamente más concentrada** de lo que produciría una asignación aleatoria.

In [ ]:
# Visualización combinada con splot
plot_moran(moran);

El panel **izquierdo** muestra la distribución empírica (gris) del I de Moran calculado para 999 mapas aleatorios simulados. La marca azul señala la media; la marca roja indica el I de Moran **observado**. El valor observado se ubica sustancialmente por encima de la distribución de aleatoriedad. El panel **derecho** replica el gráfico de dispersión de Moran.

## 3.4 Otros índices globales

Aunque el I de Moran domina las aplicaciones, existen medidas alternativas que capturan diferentes aspectos de la autocorrelación espacial.

### C de Geary

La **razón de contigüidad** de Robert Geary se expresa como:

$$C = \frac{(n-1)}{2 \sum_i \sum_j w_{ij}} \cdot \frac{\sum_i \sum_j w_{ij} (y_i - y_j)^2}{\sum_i (y_i - \bar{y})^2}$$

Donde $n$ es el número de observaciones, $w_{ij}$ indica relaciones binarias de vecindad (1 o 0), $y_i$ es el valor de la variable en la observación $i$, y $\bar{y}$ es la media muestral.

Como el I de Moran, la C de Geary compara las relaciones locales de $Y$ en el vecindario con las relaciones del total de la muestra. Sin embargo, hay diferencias clave:
- El **I de Moran** usa **productos cruzados** de valores estandarizados
- La **C de Geary** usa **diferencias** de valores no estandarizados

Valores de referencia:
- $C \approx 1$: aleatoriedad espacial
- $C < 1$: autocorrelación positiva
- $C > 1$: autocorrelación negativa

In [ ]:
geary = esda.geary.Geary(db["Pct_Leave"], w)

print(f"C de Geary: {geary.C:.4f}")
print(f"Pseudo p-valor: {geary.p_sim:.4f}")

La C de Geary confirma los resultados del I de Moran: hay una **desviación significativa de la aleatoriedad espacial**, apoyando la conclusión de autocorrelación espacial positiva en la distribución del voto Leave.

### G de Getis y Ord

El estadístico **G** propuesto originalmente por Getis y Ord representa una familia de autocorrelación espacial basada en **distancia**. Se aplica específicamente a puntos, aunque es aplicable a polígonos con pesos binarios. Se expresa como:

$$G(d) = \frac{\sum_i \sum_j w_{ij}(d) \, y_i \, y_j}{\sum_i \sum_j y_i \, y_j}$$

Donde $w_{ij}(d)$ es el peso binario asignado según criterios de banda de distancia entre las observaciones $i$ y $j$.

G mide la **concentración** más que la autocorrelación: si valores similares se co-localizan. A diferencia de Moran y Geary, **G detecta específicamente autocorrelación positiva** y no puede identificar autocorrelación negativa.

In [ ]:
# Reproyectar a OSGB (EPSG:27700) para distancias en metros
db_osgb = db.to_crs(epsg=27700)

# Calcular la distancia mínima umbral para que todos tengan al menos un vecino
pts = db_osgb.centroid
xys = pandas.DataFrame({"X": pts.x, "Y": pts.y})
min_thr = weights.util.min_threshold_distance(xys)
print(f"Distancia umbral mínima: {min_thr:.1f} metros")

In [ ]:
# Crear pesos de banda de distancia
w_db = weights.DistanceBand.from_dataframe(db_osgb, min_thr)

In [ ]:
# Calcular G de Getis y Ord
gao = esda.getisord.G(db["Pct_Leave"].values, w_db)

print(
    "Getis & Ord G: %.3f | Pseudo P-valor: %.3f" % (gao.G, gao.p_sim)
)

El pseudo p-valor sugiere una **desviación significativa** de la hipótesis de no-concentración, indicando patrones claros de concentración en la distribución espacial del voto Leave.

# 4. Resumen

| Estadístico | Tipo de variable | Qué mide | Valor de referencia |
|-------------|-----------------|----------|--------------------|
| Join Counts | Binaria | Agrupación de categorías | Esperado bajo aleatoriedad |
| I de Moran | Continua | Correlación espacial global | $I = 0$ (sin autocorrelación) |
| C de Geary | Continua | Similitud entre vecinos | $C = 1$ (aleatoriedad) |
| G de Getis-Ord | Continua (positiva) | Concentración de valores | Esperado bajo aleatoriedad |

Los tres estadísticos para variables continuas confirman la presencia de **autocorrelación espacial positiva significativa** en la distribución del voto a favor de abandonar la UE en el Reino Unido.

# 5. Preguntas de estudio

1. Vuelve a la tabla original `ref` y extrae la variable `Pct_Rejected` (votos rechazados):
   - a) Crea un mapa coroplético mostrando la distribución espacial de `Pct_Rejected`.
   - b) Construye una matriz de pesos de 8 vecinos más cercanos.
   - c) Crea un gráfico de Moran relacionando `Pct_Rejected` con su rezago espacial.
   - d) Calcula el I de Moran para `Pct_Rejected`.
   - e) Interpreta los resultados. ¿Qué aprendemos sobre la geografía del rechazo de votos?

2. A veces los referéndums requieren más del 50% para hacer efectivo el cambio. Imaginemos que el Brexit requería un 60%:
   - a) Usa `Pct_Leave` para crear una variable binaria que tome valor 1 si el porcentaje es mayor a 60, 0 en caso contrario.
   - b) Crea un mapa coroplético con la nueva variable. ¿Hay diferencias en el patrón geográfico?
   - c) Recalcula el estadístico Join Counts. ¿Qué puedes concluir? ¿Hay cambios notables?

3. Explora el efecto de diferentes matrices de pesos:
   - a) Crea dos matrices KNN adicionales: una con 4 vecinos (`wk4`) y otra con 12 (`wk12`).
   - b) Crea mapas del rezago espacial de `Pct_Leave` con cada nueva matriz. ¿Cómo difieren? ¿Por qué?
   - c) Genera gráficos de Moran con `wk4` y `wk12`. ¿Difieren del creado anteriormente?
   - d) Calcula el I de Moran con todas las matrices y compara resultados.

4. Con la misma matriz de pesos, calcula Moran, Geary y Getis-Ord para `Pct_Rejected`. ¿Llegas a conclusiones sustancialmente diferentes con cada estadístico? De ser así, ¿por qué?

5. A partir de los resultados de la pregunta 3, ¿puedes generalizar el efecto de un mayor número de vecinos en la matriz de pesos al explorar la autocorrelación espacial global?

6. ¿Es posible encontrar casos en que el I de Moran y la G de Getis-Ord estén en desacuerdo sustancial? ¿Qué podría provocar ese resultado? ¿Qué implica para el uso e interpretación de ambos estadísticos?

7. Usando pesos de k-vecinos más cercanos, ¿puedes encontrar el valor de k donde el I de Moran es mayor? Haz un gráfico de I de Moran vs. k.

8. Como en la pregunta anterior, ¿a qué valor de k la C de Geary es mayor?